# Phase 1 — PP-OCRv5_server_det: chẩn đoán lệch "dòng vs từ" trên ICDAR2015

Notebook này **không chạy lại model**. Nó đọc `paddle_icdar2015_test_predictions.jsonl` đã lưu từ notebook Phase 1 và GT ICDAR2015 test, rồi:

1. Chấm lại bằng **đúng evaluator** của notebook Phase 1 (code được copy nguyên văn) và kiểm tra ra **cùng số** P/R/Hmean với `metrics.json` của lần chạy đó.
2. Phân loại từng lỗi:
   - **FP** (box model không khớp): `FP_merged_line` (box chứa ≥ 2 từ GT — box cả dòng), `FP_localization` (chứa đúng 1 từ nhưng IoU ≤ 0,5), `FP_no_care_word` (không chứa từ GT nào: nền, hoặc chữ không được gán nhãn).
   - **FN** (từ GT bị bỏ sót): `FN_inside_merged_line` (từ đó nằm trong một box dòng), `FN_localization` (được box phủ nhưng không phải box dòng), `FN_missed` (không box nào phủ ≥ 50% từ đó).
3. Tính chỉ số **coverage** (Word Coverage Recall + tỷ lệ box nằm trên chữ) và recall theo **kích thước chữ**.

**Mọi số ở đây là diagnostic.** Số chính thức của Phase 1 vẫn là P/R/Hmean theo giao thức ICDAR2015 trong `metrics.json`. Ngưỡng của notebook này được cố định ở cell Config trước khi xem kết quả.

### Cài đặt trên Kaggle
- **Accelerator: None** (chỉ CPU), **Internet: Off** cũng được. Không cài thêm gói nào.
- Cần **CẢ HAI** input (panel Input phải hiện cả mục DATASETS lẫn NOTEBOOKS):
  1. **Add Input → Datasets**: https://www.kaggle.com/datasets/bestofbests9/icdar2015 (GT + ảnh).
  2. **Add Input → Notebooks → Your Work**: notebook Phase 1 của bạn, đúng **version đã chạy xong** (version mới nhất). Cách khác: upload `results.zip` thành một Dataset riêng.
- Ô đầu của mục 3 luôn in cây `/kaggle/input` để kiểm tra đã gắn đúng chưa.

## 0. Config

In [ ]:
import os, sys, re, json, hashlib, datetime
from pathlib import Path
from collections import Counter

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ART = WORK / 'artifacts' / 'phase1_paddle_diag'
EX_DIR = ART / 'diag_examples'
for d in (ART, EX_DIR):
    d.mkdir(parents=True, exist_ok=True)

PRED_NAME = 'paddle_icdar2015_test_predictions.jsonl'
EXPECTED_PRED_SHA256 = '4baef5ddb9c09bcbb842dbf61f83a28af3e445a62755356844894a2870716c7f'   # lần chạy 2026-10-30
PRED_FILE = None            # điền tay nếu tự dò sai

# Giống hệt notebook Phase 1 (evaluator dùng các tên này)
ICDAR15_TEST_IMG_DIR = None
ICDAR15_TEST_GT_DIR = None
ICDAR15_PADDLE_LABEL = None
EXPECTED_N_IMAGES = 500
IOU_THR = 0.5
IGNORE_AREA_PRECISION_THR = 0.5
IGNORE_TAGS = ('###', '*')

# Ngưỡng chẩn đoán — cố định trước khi xem kết quả
COVER_THR = 0.5             # từ GT "nằm trong" một box khi >= 50% diện tích từ nằm trong box đó
ON_TEXT_THR = 0.5           # box "nằm trên chữ" khi >= 50% diện tích box nằm trong hợp các polygon GT (kể cả ###)
SIZE_BINS = [0, 10, 20, 40, float('inf')]   # chiều cao từ GT (px, cạnh ngắn của hình chữ nhật xoay nhỏ nhất)
N_EXAMPLES = 12
RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat()


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

## 1. Evaluator — copy nguyên văn từ notebook Phase 1

In [ ]:
# == CORE ==
import numpy as np
from shapely.geometry import Polygon

ID_RE_GT = re.compile(r'gt_img_(\d+)\.txt')
ID_RE_IMG = re.compile(r'img_(\d+)\.(?:jpg|jpeg|png)', re.I)
LABEL_RE = re.compile(r'.*test.*label.*\.txt', re.I)


def scan_dirs(roots):
    gt_dirs, img_dirs, label_files = {}, {}, []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for dp, dn, fn in os.walk(root):
            g = {int(m.group(1)) for f in fn if (m := ID_RE_GT.fullmatch(f))}
            i = {int(m.group(1)) for f in fn if (m := ID_RE_IMG.fullmatch(f))}
            if g:
                gt_dirs[Path(dp)] = g
            if i:
                img_dirs[Path(dp)] = i
            label_files += [Path(dp) / f for f in fn if LABEL_RE.fullmatch(f)]
    return gt_dirs, img_dirs, label_files


def pick_one(cands, what):
    if len(cands) == 1:
        return cands[0]
    test_like = [c for c in cands if 'test' in str(c).lower()]
    if len(test_like) == 1:
        return test_like[0]
    raise RuntimeError(f'{what}: {len(cands)} ứng viên, hãy điền tay trong CONFIG: {[str(c) for c in cands]}')


def resolve_icdar15(roots, n_expected=EXPECTED_N_IMAGES):
    want = set(range(1, n_expected + 1))
    gt_dirs, img_dirs, label_files = scan_dirs(roots)
    if ICDAR15_TEST_IMG_DIR:
        img_dir = Path(ICDAR15_TEST_IMG_DIR)
    else:
        cands = [d for d, ids in img_dirs.items() if ids == want]
        if not cands:
            raise RuntimeError(f'Không tìm thấy thư mục chứa đúng img_1..img_{n_expected}. Các thư mục ảnh thấy được: '
                               f'{[(str(d), len(ids)) for d, ids in img_dirs.items()][:10]}')
        img_dir = pick_one(cands, 'ICDAR2015 test image dir')
    if ICDAR15_PADDLE_LABEL:
        return img_dir, Path(ICDAR15_PADDLE_LABEL), 'paddle_label'
    if ICDAR15_TEST_GT_DIR:
        return img_dir, Path(ICDAR15_TEST_GT_DIR), 'rrc'
    gt_cands = [d for d, ids in gt_dirs.items() if ids == want]
    if gt_cands:
        return img_dir, pick_one(gt_cands, 'ICDAR2015 test GT dir'), 'rrc'
    if label_files:
        return img_dir, pick_one(label_files, 'PaddleOCR label file'), 'paddle_label'
    raise RuntimeError('Không tìm thấy GT test (gt_img_*.txt hoặc *test*label*.txt). Điền tay trong CONFIG.')


def parse_rrc_gt_file(path):
    # dòng RRC: x1,y1,...,x4,y4,transcription (transcription có thể chứa dấu phẩy)
    items = []
    with open(path, encoding='utf-8-sig') as f:
        for ln, line in enumerate(f, 1):
            line = line.strip().lstrip(chr(0xFEFF))
            if not line:
                continue
            parts = line.split(',')
            if len(parts) < 9:
                raise ValueError(f'{path}:{ln}: cần 8 toạ độ + text, nhận {line!r}')
            pts = np.array([float(x) for x in parts[:8]], dtype=np.float64).reshape(4, 2)
            text = ','.join(parts[8:])
            items.append({'points': pts, 'text': text, 'ignore': text in IGNORE_TAGS})
    return items


def parse_paddle_label(path):
    gts = {}
    with open(path, encoding='utf-8') as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            rel, js = line.rstrip('\n').split('\t', 1)
            m = ID_RE_IMG.fullmatch(Path(rel).name)
            if not m:
                raise ValueError(f'{path}:{ln}: tên ảnh lạ {rel!r}')
            gts[int(m.group(1))] = [
                {'points': np.asarray(a['points'], dtype=np.float64).reshape(-1, 2),
                 'text': a['transcription'], 'ignore': a['transcription'] in IGNORE_TAGS}
                for a in json.loads(js)]
    return gts


def load_icdar15(img_dir, gt_src, fmt, n_expected=EXPECTED_N_IMAGES):
    img_dir, gt_src = Path(img_dir), Path(gt_src)
    imgs = {int(m.group(1)): img_dir / f for f in os.listdir(img_dir) if (m := ID_RE_IMG.fullmatch(f))}
    if fmt == 'rrc':
        gts = {int(m.group(1)): parse_rrc_gt_file(gt_src / f)
               for f in os.listdir(gt_src) if (m := ID_RE_GT.fullmatch(f))}
    else:
        gts = parse_paddle_label(gt_src)
    if set(imgs) != set(gts):
        raise ValueError(f'Lệch id ảnh/GT: chỉ có ảnh={sorted(set(imgs) - set(gts))[:10]}, '
                         f'chỉ có GT={sorted(set(gts) - set(imgs))[:10]}')
    if len(imgs) != n_expected:
        raise ValueError(f'Có {len(imgs)} ảnh, kỳ vọng {n_expected} (ICDAR2015 test)')
    return [{'image_id': f'img_{k}', 'path': str(imgs[k]), 'gt': gts[k]} for k in sorted(imgs)]


def _valid_poly(pts):
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    if len(pts) < 3 or not np.all(np.isfinite(pts)):
        return None
    p = Polygon(pts)
    return p if p.is_valid else None


def evaluate_image(gt_items, det_polys, iou_thr=IOU_THR, ign_thr=IGNORE_AREA_PRECISION_THR):
    gt_pols, gt_ign, det_pols, det_ign = [], [], [], []
    n_invalid_gt = n_invalid_det = 0
    for g in gt_items:
        p = _valid_poly(g['points'])
        if p is None:
            n_invalid_gt += 1
            continue
        gt_pols.append(p)
        if g['ignore']:
            gt_ign.append(len(gt_pols) - 1)
    for d in det_polys:
        p = _valid_poly(d)
        if p is None:
            n_invalid_det += 1
            continue
        det_pols.append(p)
        for gi in gt_ign:
            if p.area > 0 and gt_pols[gi].intersection(p).area / p.area > ign_thr:
                det_ign.append(len(det_pols) - 1)
                break
    gt_ign_s, det_ign_s = set(gt_ign), set(det_ign)
    gt_hit = np.zeros(len(gt_pols), dtype=bool)
    det_hit = np.zeros(len(det_pols), dtype=bool)
    pairs = []
    if gt_pols and det_pols:
        iou = np.zeros((len(gt_pols), len(det_pols)))
        for i, g in enumerate(gt_pols):
            for j, d in enumerate(det_pols):
                inter = g.intersection(d).area
                if inter > 0:
                    iou[i, j] = inter / g.union(d).area
        for i in range(len(gt_pols)):
            for j in range(len(det_pols)):
                if (not gt_hit[i] and not det_hit[j] and i not in gt_ign_s and j not in det_ign_s
                        and iou[i, j] > iou_thr):
                    gt_hit[i] = det_hit[j] = True
                    pairs.append((i, j, float(iou[i, j])))
    return {
        'n_gt_care': len(gt_pols) - len(gt_ign), 'n_det_care': len(det_pols) - len(det_ign),
        'matched': len(pairs), 'n_gt_ignore': len(gt_ign), 'n_det_ignore': len(det_ign),
        'n_invalid_gt': n_invalid_gt, 'n_invalid_det': n_invalid_det, 'pairs': pairs,
        'gt_pols': gt_pols, 'det_pols': det_pols, 'gt_ign': gt_ign_s, 'det_ign': det_ign_s,
        'gt_hit': gt_hit, 'det_hit': det_hit,
    }


def combine(results):
    m = sum(r['matched'] for r in results)
    g = sum(r['n_gt_care'] for r in results)
    d = sum(r['n_det_care'] for r in results)
    P = m / d if d else 0.0
    R = m / g if g else 0.0
    H = 2 * P * R / (P + R) if P + R else 0.0
    return {'precision': P, 'recall': R, 'hmean': H, 'matched': m, 'gt_care': g, 'det_care': d,
            'gt_ignore': sum(r['n_gt_ignore'] for r in results),
            'det_ignore': sum(r['n_det_ignore'] for r in results),
            'invalid_gt': sum(r['n_invalid_gt'] for r in results),
            'invalid_det': sum(r['n_invalid_det'] for r in results)}

## 2. Chẩn đoán

Với mỗi ảnh, dùng đúng kết quả ghép cặp của evaluator (`gt_hit`, `det_hit`, `gt_ign`, `det_ign`), rồi:
- `cov[i, j]` = phần diện tích của từ GT `i` nằm trong box `j`.
- Box `j` chứa từ `i` khi `cov[i, j] >= COVER_THR`.

In [ ]:
# == DIAG CORE ==
from shapely.ops import unary_union


def word_height(p):
    r = p.minimum_rotated_rectangle
    pts = np.asarray(r.exterior.coords, dtype=np.float64)[:-1]
    if len(pts) < 3:
        return 0.0
    edges = [float(np.hypot(*(pts[(k + 1) % len(pts)] - pts[k]))) for k in range(len(pts))]
    return min(e for e in edges if e > 0) if any(e > 0 for e in edges) else 0.0


def diagnose_image(r):
    gt, det = r['gt_pols'], r['det_pols']
    care_g = [i for i in range(len(gt)) if i not in r['gt_ign']]
    care_d = [j for j in range(len(det)) if j not in r['det_ign']]
    cov = np.zeros((len(gt), len(det)))
    for i in care_g:
        for j in range(len(det)):
            inter = gt[i].intersection(det[j]).area
            if inter > 0:
                cov[i, j] = inter / gt[i].area
    words_in = {j: [i for i in care_g if cov[i, j] >= COVER_THR] for j in range(len(det))}
    all_det = unary_union(det) if det else None
    all_gt = unary_union(gt) if gt else None

    det_rows = []
    for j in care_d:
        n = len(words_in[j])
        on_text = det[j].intersection(all_gt).area / det[j].area if all_gt is not None and det[j].area > 0 else 0.0
        if r['det_hit'][j]:
            cat = 'TP'
        elif n >= 2:
            cat = 'FP_merged_line'
        elif n == 1:
            cat = 'FP_localization'
        else:
            cat = 'FP_no_care_word'
        det_rows.append({'det_idx': j, 'category': cat, 'n_care_words_inside': n, 'on_text_frac': float(on_text)})

    gt_rows = []
    for i in care_g:
        cu = gt[i].intersection(all_det).area / gt[i].area if all_det is not None else 0.0
        if r['gt_hit'][i]:
            cat = 'TP'
        elif cu < COVER_THR:
            cat = 'FN_missed'
        else:
            jb = int(np.argmax(cov[i]))
            cat = 'FN_inside_merged_line' if cov[i, jb] >= COVER_THR and len(words_in[jb]) >= 2 else 'FN_localization'
        gt_rows.append({'gt_idx': i, 'category': cat, 'union_coverage': float(cu), 'height_px': word_height(gt[i])})
    return det_rows, gt_rows


def summarize(det_df, gt_df):
    n_det, n_gt = len(det_df), len(gt_df)
    tp = int((gt_df['category'] == 'TP').sum())
    fp = det_df[det_df['category'] != 'TP']
    fn = gt_df[gt_df['category'] != 'TP']
    cov_r = float((gt_df['union_coverage'] >= COVER_THR).mean()) if n_gt else 0.0
    on_text_p = float((det_df['on_text_frac'] >= ON_TEXT_THR).mean()) if n_det else 0.0
    return {
        'n_det_care': n_det, 'n_gt_care': n_gt, 'tp': tp, 'fp': len(fp), 'fn': len(fn),
        'fp_breakdown': fp['category'].value_counts().to_dict(),
        'fn_breakdown': fn['category'].value_counts().to_dict(),
        'word_coverage_recall': cov_r,
        'det_on_text_precision': on_text_p,
        'coverage_hmean': 2 * cov_r * on_text_p / (cov_r + on_text_p) if cov_r + on_text_p else 0.0,
        'recall_upper_bound_if_lines_split_into_words':
            (tp + int((gt_df['category'] == 'FN_inside_merged_line').sum())) / n_gt if n_gt else 0.0,
    }

## 3. Nạp GT + predictions, kiểm tra toàn vẹn

In [ ]:
import pandas as pd

import zipfile


def extract_from_zips(root, dest):
    # Output notebook có thể chỉ được mount dưới dạng .zip — giải nén mọi zip chứa predictions (kể cả zip lồng zip)
    hits, queue, k = [], sorted(Path(root).rglob('*.zip')), 0
    while queue:
        z = queue.pop(0)
        k += 1
        try:
            zf = zipfile.ZipFile(z)
        except zipfile.BadZipFile:
            continue
        with zf:
            names = zf.namelist()
            if any(n.endswith(PRED_NAME) for n in names):
                out = dest / f'zip_{k}_{z.stem}'
                zf.extractall(out)
                hits += sorted(out.rglob(PRED_NAME))
            for n in names:
                if n.endswith('.zip'):
                    inner = dest / f'nested_{k}_{len(queue)}' / Path(n).name
                    inner.parent.mkdir(parents=True, exist_ok=True)
                    inner.write_bytes(zf.read(n))
                    queue.append(inner)
    return hits


def input_tree(root, max_depth=5, max_lines=60):
    lines = []
    for dp, dn, fn in os.walk(root):
        dn.sort()
        depth = len(Path(dp).relative_to(root).parts)
        if depth > max_depth:
            dn[:] = []
            continue
        lines.append('  ' * depth + f'{Path(dp).name}/  ({len(fn)} files) {sorted(fn)[:4]}')
    return '\n'.join(lines[:max_lines])


print(f'== {INPUT_ROOT}\n{input_tree(INPUT_ROOT)}\n')
try:
    IMG_DIR, GT_SRC, GT_FMT = resolve_icdar15([INPUT_ROOT])
except RuntimeError as e:
    raise RuntimeError(f'{e}\nTHIẾU dataset ICDAR2015 test: Add Input → Datasets → https://www.kaggle.com/datasets/bestofbests9/icdar2015\n'
                       'Notebook này cần CẢ HAI input: dataset ICDAR2015 và output của notebook Phase 1.') from None
samples = load_icdar15(IMG_DIR, GT_SRC, GT_FMT)
print('images :', IMG_DIR, '\nGT     :', GT_SRC)

if PRED_FILE:
    pred_path = Path(PRED_FILE)
else:
    found = sorted(INPUT_ROOT.rglob(PRED_NAME))
    if not found:
        found = extract_from_zips(INPUT_ROOT, WORK / 'from_zip')
        if found:
            print('predictions lấy từ file zip trong input:', found)
    if not found:
        raise RuntimeError(f'Không thấy {PRED_NAME} trong {INPUT_ROOT} (kể cả bên trong file .zip).\n'
                           'Cách 1: Add Input → Notebooks → Your Work → notebook Phase 1, chọn đúng version ĐÃ CHẠY XONG '
                           '(version mới nhất, không phải Version 1 bị lỗi).\n'
                           'Cách 2: Datasets → New Dataset → upload results.zip, rồi Add Input dataset đó.\n'
                           f'Hiện có trong {INPUT_ROOT}:\n{input_tree(INPUT_ROOT)}')
    by_sha = {}
    for f in found:
        by_sha.setdefault(sha256_file(f), []).append(f)
    if len(by_sha) > 1:
        raise RuntimeError(f'Thấy nhiều file predictions KHÁC nhau, điền PRED_FILE: { {k[:12]: [str(x) for x in v] for k, v in by_sha.items()} }')
    pred_path = next(iter(by_sha.values()))[0]
PRED_SHA = sha256_file(pred_path)
print('predictions:', pred_path, '| sha256', PRED_SHA)
if PRED_SHA != EXPECTED_PRED_SHA256:
    print('WARN: sha256 khác lần chạy 2026-10-30 — đang chẩn đoán một bộ predictions khác')

preds = {}
with open(pred_path, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            if p['image_id'] in preds:
                raise ValueError(f'trùng image_id {p["image_id"]}')
            preds[p['image_id']] = p
assert set(preds) == {s['image_id'] for s in samples}, 'predictions và GT không cùng tập ảnh'

# cùng bytes ảnh với lần chạy Phase 1?
run_dir = pred_path.parent
manifest = run_dir / 'dataset_manifest.tsv'
if manifest.exists():
    old = {r['image_id']: r['sha256'] for r in pd.read_csv(manifest, sep='\t').to_dict('records')}
    diff = [s['image_id'] for s in samples if sha256_file(s['path']) != old.get(s['image_id'])]
    print('ảnh khác bytes so với lần chạy Phase 1:', len(diff), diff[:5])
    assert not diff, 'ảnh test không trùng bytes với lần chạy Phase 1'
else:
    print('WARN: không có dataset_manifest.tsv cạnh predictions — bỏ qua so khớp bytes ảnh')

## 4. Chấm lại (phải ra đúng số của Phase 1) + chẩn đoán từng ảnh

In [ ]:
eval_results, det_all, gt_all = {}, [], []
for s in samples:
    iid = s['image_id']
    r = evaluate_image(s['gt'], [np.asarray(x).reshape(-1, 2) for x in preds[iid]['polygons']])
    eval_results[iid] = r
    d_rows, g_rows = diagnose_image(r)
    det_all += [{'image_id': iid, **x} for x in d_rows]
    gt_all += [{'image_id': iid, **x} for x in g_rows]
PRIMARY = combine(list(eval_results.values()))
det_df, gt_df = pd.DataFrame(det_all), pd.DataFrame(gt_all)
print(f'chấm lại: P={PRIMARY["precision"]:.4f} R={PRIMARY["recall"]:.4f} H={PRIMARY["hmean"]:.4f}')

metrics_path = run_dir / 'metrics.json'
if metrics_path.exists():
    ref = json.loads(metrics_path.read_text(encoding='utf-8'))['primary']
    for k in ('precision', 'recall', 'hmean', 'matched', 'gt_care', 'det_care'):
        assert abs(PRIMARY[k] - ref[k]) < 1e-12, f'{k}: {PRIMARY[k]} != {ref[k]} (metrics.json Phase 1)'
    print('khớp metrics.json của Phase 1: OK')
else:
    print('WARN: không có metrics.json cạnh predictions — không so được số Phase 1')

# kiểm tra chéo: số TP/FP/FN của phần chẩn đoán phải khớp evaluator
assert int((det_df['category'] == 'TP').sum()) == PRIMARY['matched'] == int((gt_df['category'] == 'TP').sum())
assert len(det_df) == PRIMARY['det_care'] and len(gt_df) == PRIMARY['gt_care']
SUMMARY = summarize(det_df, gt_df)
print(json.dumps(SUMMARY, indent=2, ensure_ascii=False))

## 5. Recall theo kích thước chữ

In [ ]:
gt_df['size_bin'] = pd.cut(gt_df['height_px'], SIZE_BINS, right=False,
                           labels=[f'{int(a)}-{b if b == float("inf") else int(b)}px' for a, b in zip(SIZE_BINS, SIZE_BINS[1:])])
size_tbl = gt_df.groupby('size_bin', observed=False).agg(
    n_words=('category', 'size'),
    iou_recall=('category', lambda c: (c == 'TP').mean()),
    coverage_recall=('union_coverage', lambda u: (u >= COVER_THR).mean()),
    missed=('category', lambda c: (c == 'FN_missed').mean()),
    inside_merged_line=('category', lambda c: (c == 'FN_inside_merged_line').mean()),
).round(4)
print(size_tbl)

## 6. Ảnh minh hoạ lỗi

GT: **xanh lá** = TP · **vàng** = FN nằm trong box dòng · **xanh ngọc** = FN do lệch vị trí · **cam** = FN bỏ sót · **xám** = `###`.
Box model (nét mảnh): **xanh dương** = TP · **tím** = FP box dòng · **hồng** = FP lệch vị trí · **đỏ** = FP không chứa từ GT · **xám** = bị ignore.
Chọn ảnh: một nửa có nhiều `FP_merged_line` nhất, một nửa có nhiều `FN_missed` nhất.

In [ ]:
import cv2

GT_COL = {'TP': (0, 200, 0), 'FN_inside_merged_line': (0, 230, 255), 'FN_localization': (255, 255, 0), 'FN_missed': (0, 140, 255)}
DET_COL = {'TP': (255, 80, 0), 'FP_merged_line': (255, 0, 200), 'FP_localization': (180, 120, 255), 'FP_no_care_word': (0, 0, 255)}
GRAY = (160, 160, 160)


def ring(p):
    return np.asarray(p.exterior.coords, dtype=np.float64)[:-1].round().astype(np.int32).reshape(-1, 1, 2)


def draw_diag(iid):
    s = by_id[iid]
    r = eval_results[iid]
    img = cv2.imread(s['path'])
    gcat = dict(zip(gt_df.loc[gt_df.image_id == iid, 'gt_idx'], gt_df.loc[gt_df.image_id == iid, 'category']))
    dcat = dict(zip(det_df.loc[det_df.image_id == iid, 'det_idx'], det_df.loc[det_df.image_id == iid, 'category']))
    for i, p in enumerate(r['gt_pols']):
        cv2.polylines(img, [ring(p)], True, GT_COL.get(gcat.get(i), GRAY), 3)
    for j, p in enumerate(r['det_pols']):
        cv2.polylines(img, [ring(p)], True, DET_COL.get(dcat.get(j), GRAY), 1)
    c = Counter(dcat.values()) + Counter(gcat.values())
    txt = f'{iid} TP={c["TP"] // 2} FPline={c["FP_merged_line"]} FNline={c["FN_inside_merged_line"]} FNmiss={c["FN_missed"]}'
    cv2.putText(img, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
    cv2.putText(img, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return img


by_id = {s['image_id']: s for s in samples}
n_line = det_df[det_df.category == 'FP_merged_line'].groupby('image_id').size().sort_values(ascending=False)
n_miss = gt_df[gt_df.category == 'FN_missed'].groupby('image_id').size().sort_values(ascending=False)
chosen = list(n_line.index[:N_EXAMPLES // 2])
chosen += [i for i in n_miss.index if i not in chosen][:N_EXAMPLES - len(chosen)]
for iid in chosen:
    tag = 'line' if iid in n_line.index[:N_EXAMPLES // 2] else 'missed'
    cv2.imwrite(str(EX_DIR / f'{tag}_{iid}.jpg'), draw_diag(iid))
print(len(chosen), 'ảnh ->', EX_DIR)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, iid in zip(axes.ravel(), chosen[:2] + chosen[N_EXAMPLES // 2:N_EXAMPLES // 2 + 2]):
    ax.imshow(cv2.cvtColor(draw_diag(iid), cv2.COLOR_BGR2RGB))
    ax.set_title(iid)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Lưu kết quả + báo cáo

In [ ]:
det_df.to_csv(ART / 'diag_det_items.csv', index=False)
gt_df.to_csv(ART / 'diag_gt_items.csv', index=False)
size_tbl.to_csv(ART / 'diag_recall_by_word_height.csv')
DIAG = {
    'kind': 'DIAGNOSTIC — không thay thế số chính thức trong metrics.json Phase 1',
    'predictions_file': str(pred_path), 'predictions_sha256': PRED_SHA,
    'gt_source': str(GT_SRC), 'primary_rescored': PRIMARY,
    'thresholds': {'COVER_THR': COVER_THR, 'ON_TEXT_THR': ON_TEXT_THR, 'SIZE_BINS': [str(b) for b in SIZE_BINS]},
    'summary': SUMMARY,
    'recall_by_word_height': size_tbl.reset_index().astype({'size_bin': str}).to_dict('records'),
    'utc_time': RUN_STARTED,
}
write_json(ART / 'diag_metrics.json', DIAG)


def pct(x):
    return f'{100 * x:.2f}'


def md_table(df):
    cols = [df.index.name or ''] + list(df.columns)
    rows = ['| ' + ' | '.join(map(str, cols)) + ' |', '|' + '---|' * len(cols)]
    rows += ['| ' + ' | '.join([str(idx)] + [str(v) for v in r]) + ' |' for idx, r in zip(df.index, df.values)]
    return '\n'.join(rows)


fpb, fnb = SUMMARY['fp_breakdown'], SUMMARY['fn_breakdown']
report = f"""# PP-OCRv5_server_det trên ICDAR2015 test — chẩn đoán lệch dòng/từ (DIAGNOSTIC)

Predictions: `{pred_path.name}` (sha256 `{PRED_SHA[:16]}…`). Chấm lại bằng đúng evaluator Phase 1:
**P {pct(PRIMARY['precision'])} · R {pct(PRIMARY['recall'])} · Hmean {pct(PRIMARY['hmean'])}** (khớp `metrics.json`).

## Phân rã lỗi (chỉ tính box/từ "care")
| Loại | Số lượng | Tỷ lệ |
|---|---:|---:|
| FP tổng | {SUMMARY['fp']} | 100% FP |
| FP_merged_line (box chứa ≥ 2 từ GT) | {fpb.get('FP_merged_line', 0)} | {pct(fpb.get('FP_merged_line', 0) / max(SUMMARY['fp'], 1))}% |
| FP_localization (chứa 1 từ, IoU ≤ 0,5) | {fpb.get('FP_localization', 0)} | {pct(fpb.get('FP_localization', 0) / max(SUMMARY['fp'], 1))}% |
| FP_no_care_word | {fpb.get('FP_no_care_word', 0)} | {pct(fpb.get('FP_no_care_word', 0) / max(SUMMARY['fp'], 1))}% |
| FN tổng | {SUMMARY['fn']} | 100% FN |
| FN_inside_merged_line | {fnb.get('FN_inside_merged_line', 0)} | {pct(fnb.get('FN_inside_merged_line', 0) / max(SUMMARY['fn'], 1))}% |
| FN_localization | {fnb.get('FN_localization', 0)} | {pct(fnb.get('FN_localization', 0) / max(SUMMARY['fn'], 1))}% |
| FN_missed (không box nào phủ ≥ {int(COVER_THR * 100)}%) | {fnb.get('FN_missed', 0)} | {pct(fnb.get('FN_missed', 0) / max(SUMMARY['fn'], 1))}% |

## Chỉ số coverage (diagnostic)
- Word Coverage Recall: **{pct(SUMMARY['word_coverage_recall'])}%** (từ GT được hợp các box phủ ≥ {int(COVER_THR * 100)}%)
- Box nằm trên chữ: **{pct(SUMMARY['det_on_text_precision'])}%** (≥ {int(ON_TEXT_THR * 100)}% diện tích box nằm trong polygon GT, kể cả ###)
- Coverage Hmean: {pct(SUMMARY['coverage_hmean'])}%
- Cận trên recall nếu mọi box dòng được tách đúng thành từ: {pct(SUMMARY['recall_upper_bound_if_lines_split_into_words'])}%

## Recall theo chiều cao chữ
{md_table(size_tbl)}

## Giới hạn
- Ngưỡng 0,5 của coverage là quy ước của notebook này, không phải giao thức benchmark nào.
- Box to có thể làm coverage recall tăng ảo; vì vậy luôn đọc kèm tỷ lệ "box nằm trên chữ".
- Không thay số chính thức: Phase 1 vẫn báo P/R/Hmean theo giao thức ICDAR2015.
"""
(ART / 'diag_report.md').write_text(report, encoding='utf-8')
print(report)

import shutil
zip_path = shutil.make_archive(str(WORK / 'phase1_paddle_diag'), 'zip', root_dir=ART.parent.parent, base_dir='artifacts')
print('zip ->', zip_path, '| sha256', sha256_file(zip_path))